In [ ]:
from rdflib import Graph, URIRef, Literal, Namespace, ConjunctiveGraph, Dataset
from rdflib.namespace import RDF, RDFS
from lxml import etree

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# load xml from https://cwe.mitre.org/data/xml/cwec_latest.xml.zip
path_xml = r""

In [ ]:
CWE = Namespace("https://cwe.mitre.org/data/definitions/")
CWEV = Namespace("https://cwe.mitre.org/views/")
SCHEMA = Namespace("http://schema.org/")

In [ ]:
# create the ontology 

ns = {"cwe": "http://cwe.mitre.org/cwe-7"}
tree  = etree.parse(path_xml)
root = tree.getroot()

ds = Dataset()

ds.add((CWE.Weakness, RDF.type, SCHEMA.Thing))
# create all weaknesses with relations

weaknesses = root.xpath("//cwe:Weaknesses/cwe:Weakness", namespaces=ns)
for weakness in weaknesses:
    # add CWE to default graph
    cwe_uri = URIRef(CWE[weakness.get("ID")])
    ds.add([cwe_uri, RDF.type, CWE.Weakness])
    ds.add([cwe_uri, RDFS.label, Literal(weakness.get("Name"))])
    print(f"added CWE {weakness.get("ID")} to default graph.")
    
    for rweak in weakness.xpath(".//cwe:Related_Weaknesses/cwe:Related_Weakness", namespaces=ns):
        view_id = rweak.get("View_ID")
        relation = rweak.get("Nature")
        rel_cwe = rweak.get("CWE_ID")
        
        # add related cwe to graph
        if relation == "ChildOf":
            ds.add([cwe_uri, SCHEMA.child_of, URIRef(CWE[rel_cwe]), URIRef(CWEV[view_id])])
            print(f"added relation {relation} with {rel_cwe} to view {view_id}")

In [ ]:
# save ontology
ds.serialize(destination="named_graphs_cwe.trig", format="trig")

In [ ]:
# load ontology
ds  = Dataset()
ds.parse(".\\metadata\\named_graphs_cwe.trig")

In [ ]:
# with view

def get_children_onto(cwe_id, ontology, view):
    # manually filter for view bc quads doesn't seem to filter 
    children = [q[0].split("/")[-1] for q in ontology.quads((None, URIRef('http://schema.org/child_of'), URIRef(f'https://cwe.mitre.org/data/definitions/{cwe_id}'), URIRef(f'https://cwe.mitre.org/views/{view}'))) if q[-1].split("/")[-1] == f"{view}"]
    new_c = []
    for child in children:
        children_new = get_children_onto(child, ontology, view)
        new_c.extend(children_new)
    children.extend(new_c)
    return children

def get_parents_onto(cwe_id, ontology, view):
    # manually filter for view bc quads doesn't seem to filter 
    parents = [q[2].split("/")[-1] for q in ontology.quads((URIRef(f'https://cwe.mitre.org/data/definitions/{cwe_id}'), URIRef('http://schema.org/child_of'), None, URIRef(f'https://cwe.mitre.org/views/{view}'))) if q[-1].split("/")[-1] == f"{view}"]    
    new_p = []
    for parent in parents:
        parents_new = get_parents_onto(parent, ontology, view)
        new_p.extend(parents_new)
    parents.extend(new_p)
    return parents


# without view

def get_children_ontov(cwe_id, ontology):
    children = [q[0].split("/")[-1] for q in ontology.quads((None, URIRef('http://schema.org/child_of'), URIRef(f'https://cwe.mitre.org/data/definitions/{cwe_id}'), None))]
    new_c = []
    for child in children:
        children_new = get_children_ontov(child, ontology)
        new_c.extend(children_new)
    children.extend(new_c)
    return children

def get_parents_ontov(cwe_id, ontology):
    parents = [q[2].split("/")[-1] for q in ontology.quads((URIRef(f'https://cwe.mitre.org/data/definitions/{cwe_id}'), URIRef('http://schema.org/child_of'), None, None))]    
    new_p = []
    for parent in parents:
        parents_new = get_parents_ontov(parent, ontology)
        new_p.extend(parents_new)
    parents.extend(new_p)
    return parents